# Cat Boost Ablation (Proof of Concept w/ Library)

# Setup

In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

from preprocessing import clean_data, CATEGORICAL_FEATURES
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/cat_boost.csv"

########## DEBUG ##########
DEBUG = True

########## MODEL ##########
INNER_CV = 2
OUTER_CV = 2
SCORING = 'neg_root_mean_squared_error'
MAX_FEATURES = 'sqrt'
RANDOM_STATE = 0

# Param Grid
DEPTH = [10]
LEARNING_RATE = [0.05, 0.1]
LEAF_REG = [3, 5, 7]
ITERATIONS = [1500]
BAGGING_TEMPERATURE = [0.5, 1, 2]
BORDER_COUNT = [254]

# Init & Pre-Processing

In [3]:
data = pd.read_csv (TRAIN_PATH)
labels, cleaned_data = clean_data (data)

# No drop
def engineer_data (
    cl_d: pd.DataFrame
) -> pd.DataFrame:
    """
    Feature Engineering
    Returns engineered dataframe
    """
    # Convert date to monthly circular representation
    months = pd.to_datetime (cl_d['Date']).dt.month
    engineered_data = cl_d.drop (columns = ['Date'])
    engineered_data['Month_sin'] = np.sin (2 * np.pi * months / 12)
    engineered_data['Month_cos'] = np.cos (2 * np.pi * months / 12)

    return engineered_data

feature_engineer = FunctionTransformer (engineer_data)

# Model

In [4]:
# Pipeline
pipeline = Pipeline ([('f_eng', feature_engineer),
                      ('cat', CatBoostRegressor (
                                    task_type = 'GPU',
                                    loss_function = 'RMSE',
                                    cat_features = CATEGORICAL_FEATURES,
                                    random_seed = 0))])

# Grid Search Double CV
param_grid = {'cat__depth': DEPTH,
              'cat__learning_rate': LEARNING_RATE,
              'cat__l2_leaf_reg': LEAF_REG,
              'cat__iterations': ITERATIONS,
              'cat__bagging_temperature': BAGGING_TEMPERATURE,
              'cat__border_count': BORDER_COUNT}

gs = GridSearchCV (estimator = pipeline,
                   param_grid = param_grid,
                   scoring = SCORING,
                   cv = INNER_CV,
                   n_jobs = -1)

nested_scores = cross_val_score (gs,
                                 cleaned_data,
                                 labels,
                                 cv = OUTER_CV,
                                 scoring = SCORING)

# Evaluate
print (f"Nested RMSE: {-nested_scores.mean ()}")
if (DEBUG):
    print (f"Fold RMSEs: {-nested_scores}")

Fatal Python error: Bus error

Thread 0x00007fe5538be740 (most recent call first):
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/catboost/core.py", line 1790 in _train
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/catboost/core.py", line 2410 in _fit
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/catboost/core.py", line 5873 in fit
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/pipeline.py", line 663 in fit
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/base.py", line 1365 in wrapper
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859 in _fit_and_score
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/utils/parallel.py", line 147 in __call__
  File "/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/joblib/parallel.py", line 607 in __call__
  File "/home/osten/.virt

0:	learn: 5.2828789	total: 16s	remaining: 6h 40m 53s
1:	learn: 5.2151259	total: 16.2s	remaining: 3h 22m 16s
2:	learn: 5.1548147	total: 16.3s	remaining: 2h 15m 57s
3:	learn: 5.0960599	total: 16.5s	remaining: 1h 42m 34s
4:	learn: 5.0403902	total: 16.6s	remaining: 1h 22m 37s
0:	learn: 5.1923410	total: 18.8s	remaining: 7h 49m 42s
1:	learn: 5.0651646	total: 18.9s	remaining: 3h 55m 59s
2:	learn: 4.9614586	total: 19s	remaining: 2h 38m 1s
3:	learn: 4.8667959	total: 19.1s	remaining: 1h 58m 58s
4:	learn: 4.7843286	total: 19.2s	remaining: 1h 35m 31s
5:	learn: 4.7127638	total: 19.3s	remaining: 1h 19m 55s
6:	learn: 4.6508427	total: 19.3s	remaining: 1h 8m 46s
7:	learn: 4.5977966	total: 19.4s	remaining: 1h 24s
8:	learn: 4.5542016	total: 19.5s	remaining: 53m 52s
5:	learn: 4.9895989	total: 33.1s	remaining: 2h 17m 17s
6:	learn: 4.9406152	total: 33.2s	remaining: 1h 57m 56s
7:	learn: 4.8966602	total: 33.3s	remaining: 1h 43m 30s
8:	learn: 4.8556577	total: 33.4s	remaining: 1h 32m 16s
9:	learn: 4.8190596	tot

Application terminated with error: ??+0 (0x7FA13F6D9D8A)
??+0 (0x7FA13EF5ABA4)
??+0 (0x7FA14060BED5)
??+0 (0x7FA14060BDAD)
??+0 (0x7FA140606CC7)
??+0 (0x7FA140605DC2)
??+0 (0x7FA140607448)
??+0 (0x7FA13F25AE21)
??+0 (0x7FA13F25AC8A)
??+0 (0x7FA1528BAAA4)
??+0 (0x7FA152947C6C)

(TCatBoostException) catboost/cuda/cuda_lib/cuda_base.h:183: CUDA error 2: out of memory
Application terminated with error: ??+0 (0x7FD0B9E2CD8A)
??+0 (0x7FD0B96ADBA4)
??+0 (0x7FD0BAD5EED5)
??+0 (0x7FD0BAD5EDAD)
??+0 (0x7FD0BAD59CC7)
??+0 (0x7FD0BAD58DC2)
??+0 (0x7FD0BAD5A448)
??+0 (0x7FD0B99ADE21)
??+0 (0x7FD0B99ADC8A)
??+0 (0x7FD0CD01FAA4)
??+0 (0x7FD0CD0ACC6C)

(TCatBoostException) catboost/cuda/cuda_lib/cuda_base.h:183: CUDA error 2: out of memory
Terminating due to uncaught exception 0x51d98190410    what() -> "catboost/cuda/cuda_lib/cuda_base.h:183: CUDA error 2: out of memory"
 of type TCatBoostException
Terminating due to uncaught exception 0x28e2e190410    what() -> "catboost/cuda/cuda_lib/cuda_base.h:18

KeyboardInterrupt: 

# Predict

In [ ]:
# Build final model with all training data
gs.fit (cleaned_data, labels)
print ("Best parameters:", gs.best_params_)
print ("Best inner CV score:", -gs.best_score_)

final_model = gs.best_estimator_

# Predict
test_data = pd.read_csv (TEST_PATH)
_, cleaned_test_data = clean_data (test_data)
predictions = final_model.predict (cleaned_test_data)

# Save
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (predictions) + 1),
                          'Milk_Yield_L': predictions})
out_data.to_csv (OUT_PATH, index = False)

NameError: name 'gs' is not defined